# Phase 3 - Step 2: Vessel Segmentation (7 Attention UNet)

## What Is Attention U-Net?
Attention U-Net (Oktay et al., 2018) improves upon standard U-Net by adding **Attention Gates** directly on the skip connections.

### The Problem With Standard U-Net Skip Connections
In a standard U-Net, skip connections pass ALL encoder features directly to the decoder — including irrelevant background texture. The decoder then struggles to distinguish vessel pixels from noise, especially for **thin capillaries**.

### The Attention Gate Solution
Instead of blindly passing encoder features, Attention U-Net first asks:
> *"Which spatial locations in this feature map are actually relevant to vessel prediction at this scale?"*

The gate learns to **amplify** vessel-relevant regions and **suppress** background. This is computed as:

```
Standard U-Net:     encoder ──────────────────────► decoder
                    (raw, noisy features)

Attention U-Net:    encoder ──► Attention Gate ──► decoder
                                     ↑
                             decoder query signal
                             (what does the decoder need?)
```

### scSE Attention (What We Use)
We use **Spatial and Channel Squeeze & Excitation (scSE)** attention, which applies TWO attention mechanisms:
1. **Channel attention (cSE):** Recalibrates which feature channels matter most (e.g. the green channel highlights vessels best)
2. **Spatial attention (sSE):** Recalibrates which spatial locations matter most (suppresses background, amplifies vessels)

The final output = element-wise max of both, giving richer attention than either alone.

### Why This Matters for Phase B
This is the direct predecessor to our **SCCSA skip connection** innovation in Phase B. Training Attention U-Net now gives us the baseline Dice score to beat with SCCSA.

### Training Strategy: Two-Phase Fine-Tuning
1. **Phase 1 (Warm-up, 5 epochs):** Encoder FROZEN — only decoder + attention gates learn. This prevents corrupting ImageNet weights with random decoder gradients.
2. **Phase 2 (Main training):** Encoder UNFROZEN — full fine-tuning with a lower LR for encoder vs decoder.

In [ ]:
# ============================================================
# Cell 1: Setup & Installation
# ============================================================
!pip install -q segmentation-models-pytorch albumentations


In [ ]:
# ============================================================
# Cell 2: Imports
# ============================================================
import os
import json
import random
import zipfile
import numpy as np

import pandas as pd
import cv2
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from pathlib import Path
from sklearn.metrics import roc_auc_score, average_precision_score, roc_curve, precision_recall_curve

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from torch.cuda.amp import GradScaler, autocast
import segmentation_models_pytorch as smp
import albumentations as A
from albumentations.pytorch import ToTensorV2


In [ ]:
# ============================================================
# Cell 3: Reproducibility + Device Setup
# ============================================================
def set_seed(seed=42):
    """
    Fix ALL random seeds for fully reproducible experiments.
    Critical for comparing models fairly — same data splits,
    same weight initialization order.
    """
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    os.environ['PYTHONHASHSEED'] = str(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

set_seed(42)

device = torch.device(
    'cuda' if torch.cuda.is_available() else
    'mps'  if torch.backends.mps.is_available() else
    'cpu'
)
print(f'Device : {device}')
if device.type == 'cuda':
    print(f'GPU    : {torch.cuda.get_device_name(0)}')
    print(f'VRAM   : {torch.cuda.get_device_properties(0).total_memory/1e9:.1f} GB')


In [ ]:
# ============================================================
# Cell 4: Load Preprocessed Dataset
# ============================================================
# The preprocessing notebook (01_preprocessing.ipynb) already produced:
#   dataset/
#     images/          <- preprocessed 512x512 fundus images
#     vessel_masks/    <- binary vessel masks
#     splits/
#       train.csv      <- img_path, vessel_path columns
#       val.csv
#       test.csv

zip_path    = 'DR_dataset_processed.zip'   # update to your zip name
extract_dir = 'dataset'

with zipfile.ZipFile(zip_path, 'r') as zf:
    zf.extractall(extract_dir)

BASE_DIR   = extract_dir
TRAIN_CSV  = os.path.join(BASE_DIR,  'train_split.csv')
VAL_CSV    = os.path.join(BASE_DIR,  'val_split.csv')
TEST_CSV   = os.path.join(BASE_DIR,  'test_split.csv')

# Quick check
for split, path in [('Train', TRAIN_CSV), ('Val', VAL_CSV), ('Test', TEST_CSV)]:
    df = pd.read_csv(path)
    print(f'{split:<6}: {len(df):>4} samples  | columns: {list(df.columns)}')


In [ ]:
# ============================================================
# Cell 5: Shared Modules (from src.dataset)
# ============================================================
# In production these live in src/dataset.py and src/transforms.py
# We paste them inline here so the notebook is self-contained.

# from src.dataset import RetinalDataset
# from src.transforms import get_train_transforms, get_val_transforms

"""
Retinal DR Detection - PyTorch Dataset & Augmentation Pipelines
================================================================
RetinalDataset reads image-mask pairs from CSV split files.
Augmentation uses Albumentations — geometric transforms are applied
IDENTICALLY to image AND mask via additional_targets={'mask':'mask'}.
This is critical for segmentation: if you flip the image but not the
mask, the model trains on corrupted labels and never converges.
"""


class RetinalDataset(Dataset):
    """
    PyTorch Dataset for retinal fundus images and segmentation masks.

    Reads image-mask pairs from a CSV file generated by the
    preprocessing notebook. Each row contains relative paths to the
    processed image and its corresponding vessel mask.

    Args:
        csv_file (str): Path to the CSV split file (train/val/test).
        base_dir (str): Root directory of the processed dataset.
        transform (callable): Albumentations transform pipeline.
        mask_col (str): Column name pointing to the mask path.
                        'vessel_path' for vessel segmentation.
    """

    def __init__(self, csv_file, base_dir, transform=None, mask_col='vessel_path'):
        self.df       = pd.read_csv(csv_file)
        self.base_dir = base_dir
        self.transform = transform
        self.mask_col  = mask_col

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]

        img_path  = os.path.join(self.base_dir, row['img_path'])
        mask_path = os.path.join(self.base_dir, row[self.mask_col])

        # Load image — cv2 reads BGR, convert to RGB
        image = cv2.imread(img_path)
        image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
        # dtype: uint8, shape: (H, W, 3)

        # Load mask — grayscale, binarize to 0.0 / 1.0
        # Some datasets use 255 for vessel, others use 1 — normalize both
        mask = cv2.imread(mask_path, cv2.IMREAD_GRAYSCALE)
        mask = (mask > 127).astype(np.float32)
        # dtype: float32, shape: (H, W), values: 0.0 or 1.0

        # Apply transforms — BOTH image and mask
        if self.transform is not None:
            aug   = self.transform(image=image, mask=mask)
            image = aug['image']   # Tensor (3, H, W)
            mask  = aug['mask']    # Tensor (H, W)

        # Add channel dim to mask: (H, W) -> (1, H, W)
        if isinstance(mask, torch.Tensor):
            mask = mask.unsqueeze(0)
        else:
            mask = torch.from_numpy(mask).unsqueeze(0)

        return image, mask


# ── Augmentation Pipelines ──────────────────────────────────────────────────

def get_train_transforms(img_size=512):
    """
    Training augmentation pipeline for retinal vessel segmentation.

    Geometric transforms are critical because:
    - Retina can be photographed from any angle → flips & rotations
    - Different cameras have different FOVs → scale variations
    - Patient movement causes slight distortions → elastic transforms
    - Attention U-Net specifically benefits from elastic transform because
      it learns to attend to curved, deformable vessel structures.

    IMPORTANT: additional_targets={'mask':'mask'} ensures the SAME
    geometric transform is applied to both image and mask.
    Color transforms only apply to the image — the mask is not RGB.
    """
    return A.Compose([
        A.Resize(img_size, img_size),

        # === Geometric (image AND mask) ===
        A.HorizontalFlip(p=0.5),
        A.VerticalFlip(p=0.5),
        A.RandomRotate90(p=0.5),
        A.ShiftScaleRotate(
            shift_limit=0.0625, scale_limit=0.1, rotate_limit=45,
            border_mode=cv2.BORDER_CONSTANT, p=0.5
        ),
        A.GridDistortion(num_steps=5, distort_limit=0.3, p=0.3),
        A.ElasticTransform(alpha=120, sigma=120*0.05, p=0.2),
        # ElasticTransform mimics natural eye/vessel curvature — especially
        # beneficial for attention gates that learn spatial vessel patterns

        # === Color / Illumination (image ONLY) ===
        A.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.2, hue=0.1, p=0.4),
        A.CLAHE(clip_limit=4.0, tile_grid_size=(8, 8), p=0.3),
        # CLAHE boosts local contrast — helps attention gates find faint thin vessels
        A.GaussianBlur(blur_limit=(3, 5), p=0.2),
        A.GaussNoise(var_limit=(5, 25), p=0.2),

        # === Normalize + Tensor (ALWAYS LAST) ===
        # ImageNet stats because ResNet34 encoder was pretrained on ImageNet
        A.Normalize(mean=(0.485, 0.456, 0.406), std=(0.229, 0.224, 0.225)),
        ToTensorV2(),
    ], additional_targets={'mask': 'mask'})


def get_val_transforms(img_size=512):
    """
    Validation/Test transforms — NO random augmentation.
    Only resize + normalize. Evaluating on augmented data gives
    fake metrics that do not reflect real deployment performance.
    """
    return A.Compose([
        A.Resize(img_size, img_size),
        A.Normalize(mean=(0.485, 0.456, 0.406), std=(0.229, 0.224, 0.225)),
        ToTensorV2(),
    ], additional_targets={'mask': 'mask'})


print('RetinalDataset and transforms ready')


In [ ]:
# ============================================================
# Cell 6: Shared Modules (from src.metrics)
# ============================================================

# from src.metrics import dice_coefficient, iou_score, sensitivity,
#                         specificity, precision_score, evaluate_full

"""
Retinal DR Detection - Medical Evaluation Metrics
===================================================
In medical imaging standard accuracy is NOT enough.
A model predicting 'no vessel everywhere' gets ~96% accuracy
(vessels are only ~4% of pixels) but is clinically useless.

Key metrics:
  Dice (F1):    Main segmentation metric — measures overlap
  IoU:          Stricter than Dice — penalises FP and FN more
  Sensitivity:  'Of all real vessels, how many did we detect?'
                CRITICAL — missing a vessel = missing pathology
  Specificity:  'Of all background pixels, how many correctly ignored?'
  AUC-ROC:      Discriminability across all thresholds
  AUC-PR:       Better than AUC-ROC for imbalanced data (vessels=4%)
"""


def dice_coefficient(pred, target, smooth=1e-6):
    """
    Dice = 2 * |P ∩ T| / (|P| + |T|)
    Ranges 0 (no overlap) to 1 (perfect). THE standard for medical segmentation.
    Args: pred/target — binary tensors (B, 1, H, W) or (B, H, W)
    """
    pf = pred.contiguous().view(-1).float()
    tf = target.contiguous().view(-1).float()
    intersection = (pf * tf).sum()
    return (2.0 * intersection + smooth) / (pf.sum() + tf.sum() + smooth)


def iou_score(pred, target, smooth=1e-6):
    """
    IoU (Jaccard Index) = |P ∩ T| / |P ∪ T|
    Stricter than Dice — penalizes FP and FN more heavily.
    """
    pf = pred.contiguous().view(-1).float()
    tf = target.contiguous().view(-1).float()
    intersection = (pf * tf).sum()
    union = pf.sum() + tf.sum() - intersection
    return (intersection + smooth) / (union + smooth)


def sensitivity(pred, target, smooth=1e-6):
    """
    Sensitivity (Recall) = TP / (TP + FN)
    'Of all real vessels, what fraction did the model find?'
    In clinical settings: a missed vessel = missed hemorrhage.
    """
    pf = pred.contiguous().view(-1).float()
    tf = target.contiguous().view(-1).float()
    tp = (pf * tf).sum()
    fn = (tf * (1 - pf)).sum()
    return (tp + smooth) / (tp + fn + smooth)


def specificity(pred, target, smooth=1e-6):
    """
    Specificity (TNR) = TN / (TN + FP)
    'Of all background pixels, how many correctly left as background?'
    """
    pf = pred.contiguous().view(-1).float()
    tf = target.contiguous().view(-1).float()
    tn = ((1 - pf) * (1 - tf)).sum()
    fp = (pf * (1 - tf)).sum()
    return (tn + smooth) / (tn + fp + smooth)


def precision_score(pred, target, smooth=1e-6):
    """
    Precision (PPV) = TP / (TP + FP)
    'Of everything predicted as vessel, how much was correct?'
    """
    pf = pred.contiguous().view(-1).float()
    tf = target.contiguous().view(-1).float()
    tp = (pf * tf).sum()
    fp = (pf * (1 - tf)).sum()
    return (tp + smooth) / (tp + fp + smooth)


def pixel_accuracy(pred, target):
    """Standard pixel accuracy. Included for completeness — NOT reliable for imbalanced data."""
    pf = pred.contiguous().view(-1).float()
    tf = target.contiguous().view(-1).float()
    return (pf == tf).sum().float() / tf.numel()


def compute_auc_roc(pred_probs, targets):
    """AUC-ROC from flattened numpy arrays of probabilities and binary labels."""
    try:
        return roc_auc_score(targets, pred_probs)
    except ValueError:
        return 0.0


def compute_auc_pr(pred_probs, targets):
    """AUC-PR (average precision) — better than AUC-ROC for class-imbalanced data."""
    try:
        return average_precision_score(targets, pred_probs)
    except ValueError:
        return 0.0


print('Metrics functions ready')


In [ ]:
# ============================================================
# Cell 7: Hyperparameter Configuration
# ============================================================
# All hyperparameters in ONE dict.
# To run a new experiment: only change values here,
# never hardcode values inside training functions.

CFG = dict(
    # Paths
    base_dir    = 'dataset',
    train_csv   = 'dataset/train_split.csv',
    val_csv     = 'dataset/val_split.csv',
    test_csv    = 'dataset/test_split.csv',
    results_dir = 'attention_UNet_results',

    # Model
    encoder_name    = 'resnet34',
    encoder_weights = 'imagenet',
    attention_type  = 'scse',      # spatial + channel SE attention gates
    in_channels     = 3,
    out_channels    = 1,
    img_size        = 512,

    # Training — two phases
    freeze_epochs   = 5,           # Phase 1: encoder frozen, decoder warms up
    total_epochs    = 80,          # Phase 2: full fine-tuning
    batch_size      = 16,           # reduce to 4 if CUDA OOM
    num_workers     = 4,

    # Learning rates — different for encoder vs decoder (two-phase strategy)
    lr_encoder      = 1e-5,        # low LR — fine-tune pretrained weights carefully
    lr_decoder      = 3e-4,        # higher LR — decoder trains from scratch

    weight_decay    = 1e-4,
    grad_clip       = 1.0,         # clips gradient norm — prevents NaN losses
    use_amp         = True,        # mixed precision — 2x speedup on modern GPUs
    warmup_epochs   = 3,           # linear LR warmup before cosine decay

    # Loss weights
    dice_weight     = 0.6,         # higher weight on Dice — handles class imbalance
    bce_weight      = 0.4,
    label_smoothing = 0.05,        # small label smoothing on BCE — prevents overconfidence

    # Early stopping
    patience        = 10,

    # Evaluation
    threshold       = 0.5,         # sigmoid threshold for binary prediction
    seed            = 42,
)

# Create results subfolders
for sub in ['checkpoints', 'plots', 'predictions', 'metrics']:
    os.makedirs(os.path.join(CFG['results_dir'], sub), exist_ok=True)

print('Configuration ready')
print(f'Results will be saved to: {CFG["results_dir"]}/')
print(f'Encoder : {CFG["encoder_name"]} (pretrained on {CFG["encoder_weights"]})')
print(f'Attention: {CFG["attention_type"]} (spatial + channel squeeze & excitation)')
print(f'Image size: {CFG["img_size"]}x{CFG["img_size"]}')


In [ ]:
# ============================================================
# Cell 8: Build DataLoaders
# ============================================================

train_dataset = RetinalDataset(
    CFG['train_csv'], CFG['base_dir'],
    transform=get_train_transforms(CFG['img_size']),
    mask_col='vessel_path'
)
val_dataset = RetinalDataset(
    CFG['val_csv'], CFG['base_dir'],
    transform=get_val_transforms(CFG['img_size']),
    mask_col='vessel_path'
)
test_dataset = RetinalDataset(
    CFG['test_csv'], CFG['base_dir'],
    transform=get_val_transforms(CFG['img_size']),
    mask_col='vessel_path'
)

train_loader = DataLoader(
    train_dataset, batch_size=CFG['batch_size'], shuffle=True,
    num_workers=CFG['num_workers'], pin_memory=True,
    persistent_workers=True, drop_last=True,
    worker_init_fn=lambda w: np.random.seed(CFG['seed'] + w)
    # Each DataLoader worker gets its own seed so augmentations
    # are not duplicated across workers
)
val_loader = DataLoader(
    val_dataset, batch_size=CFG['batch_size'], shuffle=False,
    num_workers=CFG['num_workers'], pin_memory=True
)
test_loader = DataLoader(
    test_dataset, batch_size=CFG['batch_size'], shuffle=False,
    num_workers=CFG['num_workers'], pin_memory=True
)

print(f'Train: {len(train_dataset)} samples | {len(train_loader)} batches')
print(f'Val  : {len(val_dataset)} samples | {len(val_loader)} batches')
print(f'Test : {len(test_dataset)} samples | {len(test_loader)} batches')

# Verify one batch
imgs, masks = next(iter(train_loader))
print(f'\nBatch check:')
print(f'  Image shape : {tuple(imgs.shape)}   (B, C, H, W)')
print(f'  Mask  shape : {tuple(masks.shape)}  (B, 1, H, W)')
print(f'  Mask  values: min={masks.min():.1f}  max={masks.max():.1f}  (should be 0.0 and 1.0)')
print(f'  Vessel ratio: {masks.mean():.4f} ({masks.mean()*100:.2f}% of pixels are vessels)')


In [ ]:
# ============================================================
# Cell 9: Build Attention U-Net Model
# ============================================================

model = smp.Unet(
    encoder_name=CFG['encoder_name'],
    # ResNet34 pretrained on ImageNet.
    # Early layers learn universal features (edges, gradients, textures)
    # that transfer well even to retinal images.

    encoder_weights=CFG['encoder_weights'],
    # 'imagenet' loads pretrained weights.
    # ~3-5% Dice improvement vs random initialization on medical images.

    decoder_attention_type=CFG['attention_type'],
    # 'scse' = Spatial + Channel Squeeze & Excitation.
    # Applied to EACH decoder block — so the model attends to
    # relevant vessels at every resolution scale (512, 256, 128, 64, 32).

    in_channels=CFG['in_channels'],    # 3 = RGB fundus image
    classes=CFG['out_channels'],       # 1 = binary vessel mask
    activation=None,                   # sigmoid applied in loss, not model
    # WHY: BCEWithLogitsLoss applies sigmoid internally in a numerically
    # stable way. Applying sigmoid twice in model + loss = training bug.
)
model = model.to(device)

total     = sum(p.numel() for p in model.parameters())
trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f'Attention U-Net (ResNet34 + scSE)')
print(f'Total params    : {total:,}')
print(f'Trainable params: {trainable:,}')

# Sanity check — forward pass
with torch.no_grad():
    dummy = torch.randn(2, 3, 512, 512).to(device)
    out   = model(dummy)
    print(f'Input  shape: {tuple(dummy.shape)}')
    print(f'Output shape: {tuple(out.shape)}  (should be [2, 1, 512, 512])')
    print(f'Output range: [{out.min():.2f}, {out.max():.2f}]  (raw logits, not yet sigmoid)')


In [ ]:
# ============================================================
# Cell 10: Loss Functions
# ============================================================
# WHY NOT JUST CROSS-ENTROPY?
# Vessels = ~4% of pixels. BCE treats every pixel equally.
# A model predicting 'all background' gets 96% accuracy but Dice=0.
# Dice Loss directly measures overlap -> forces model to find vessels.
# Combined: Dice handles imbalance, BCE provides stable pixel gradients.


class DiceLoss(nn.Module):
    """
    Dice Loss = 1 - (2*|P∩T| + smooth) / (|P| + |T| + smooth)

    Directly optimizes the Dice coefficient — the same metric we report.
    Smooth prevents division by zero on empty mask regions.
    """
    def __init__(self, smooth=1.0):
        super().__init__()
        self.smooth = smooth

    def forward(self, preds, targets):
        preds  = torch.sigmoid(preds)
        pf     = preds.view(preds.size(0), -1)
        tf     = targets.view(targets.size(0), -1)
        inter  = (pf * tf).sum(dim=1)
        dice   = (2.*inter + self.smooth) / (pf.sum(dim=1) + tf.sum(dim=1) + self.smooth)
        return 1.0 - dice.mean()


class CombinedLoss(nn.Module):
    """
    Combined Dice + BCE loss with optional label smoothing.

    Label smoothing (0.05) prevents the model from being overconfident
    on ambiguous vessel-background border pixels, improving generalization.
    In retinal images, border pixels are genuinely uncertain even for doctors.
    """
    def __init__(self, dice_weight=0.6, bce_weight=0.4, label_smoothing=0.05):
        super().__init__()
        self.dw   = dice_weight
        self.bw   = bce_weight
        self.ls   = label_smoothing
        self.dice = DiceLoss(smooth=1.0)
        # BCEWithLogitsLoss = sigmoid + BCE in one numerically stable op
        self.bce  = nn.BCEWithLogitsLoss()

    def forward(self, preds, targets):
        # Apply label smoothing: 0.0 -> 0.05,  1.0 -> 0.95
        # Only applied to BCE; Dice uses raw binary targets for clean overlap measure
        smooth_targets = targets * (1 - self.ls) + 0.5 * self.ls
        d    = self.dice(preds, targets)
        b    = self.bce(preds, smooth_targets)
        loss = self.dw * d + self.bw * b
        return loss, d, b


criterion = CombinedLoss(
    dice_weight=CFG['dice_weight'],
    bce_weight=CFG['bce_weight'],
    label_smoothing=CFG['label_smoothing']
)
print('Loss function: Dice + BCE with label smoothing')
print(f'  Dice weight     : {CFG["dice_weight"]}')
print(f'  BCE weight      : {CFG["bce_weight"]}')
print(f'  Label smoothing : {CFG["label_smoothing"]}')


In [ ]:
# ============================================================
# Cell 11: Optimizer + Scheduler (Two-Phase Strategy)
# ============================================================
# Two-phase training:
#   Phase 1 (freeze_epochs): encoder frozen, only decoder + attention learn
#     -> prevents random decoder gradients from corrupting ImageNet encoder weights
#   Phase 2 (remaining epochs): encoder unfrozen with lr_encoder << lr_decoder
#     -> fine-tunes encoder carefully while decoder continues learning

def freeze_encoder(model):
    """Freeze all encoder parameters. Only decoder + attention gates train."""
    for param in model.encoder.parameters():
        param.requires_grad = False
    frozen = sum(p.numel() for p in model.encoder.parameters())
    print(f'Encoder FROZEN ({frozen:,} params frozen)')


def unfreeze_encoder(model):
    """Unfreeze encoder for full fine-tuning with lower learning rate."""
    for param in model.encoder.parameters():
        param.requires_grad = True
    freed = sum(p.numel() for p in model.encoder.parameters())
    print(f'Encoder UNFROZEN ({freed:,} params now trainable)')


def build_optimizer(model, phase=1):
    """
    Build AdamW optimizer with differential learning rates.
    Phase 1: only decoder params (encoder frozen)
    Phase 2: encoder (low LR) + decoder (high LR) — separate param groups
    """
    if phase == 1:
        # Only decoder and decoder-attention parameters
        decoder_params = list(model.decoder.parameters()) + list(model.segmentation_head.parameters())
        return optim.AdamW(decoder_params, lr=CFG['lr_decoder'], weight_decay=CFG['weight_decay'])
    else:
        # Differential LR: encoder gets 10x lower LR than decoder
        return optim.AdamW([
            {'params': model.encoder.parameters(), 'lr': CFG['lr_encoder']},
            {'params': model.decoder.parameters(), 'lr': CFG['lr_decoder']},
            {'params': model.segmentation_head.parameters(), 'lr': CFG['lr_decoder']},
        ], weight_decay=CFG['weight_decay'])


def build_scheduler(optimizer, total_epochs, warmup_epochs):
    """
    Linear warmup then cosine annealing.
    Warmup prevents large LR from destabilizing early training.
    Cosine decay smoothly reduces LR — better than step decay for segmentation.
    """
    def lr_lambda(epoch):
        if epoch < warmup_epochs:
            return (epoch + 1) / warmup_epochs          # linear warmup
        progress = (epoch - warmup_epochs) / max(1, total_epochs - warmup_epochs)
        return 0.5 * (1 + np.cos(np.pi * progress))    # cosine decay
    return optim.lr_scheduler.LambdaLR(optimizer, lr_lambda)


print('Optimizer and scheduler functions ready')


In [ ]:
# ============================================================
# Cell 12: Training + Validation Loops
# ============================================================

def train_one_epoch(model, loader, optimizer, criterion, scaler, epoch):
    """
    Train for one epoch. Returns dict with loss and metrics.

    Key correctness rules followed here:
    - model.train() called first — activates dropout + BN training stats
    - optimizer.zero_grad() before backward — no gradient accumulation bug
    - AMP autocast wraps only forward + loss — not backward
    - Gradient clip before optimizer.step() — prevents NaN from exploding grads
    - metrics computed on detached predictions — no memory leak
    """
    model.train()
    total_loss = total_dice = total_bce = 0.0
    all_dice   = []
    n = len(loader)

    for step, (images, masks) in enumerate(loader):
        images = images.to(device, non_blocking=True)
        masks  = masks.to(device,  non_blocking=True)

        optimizer.zero_grad()

        # AMP: float16 forward pass for 2x speedup
        with autocast(enabled=(scaler is not None)):
            preds = model(images)
            loss, dloss, bloss = criterion(preds, masks)


        if scaler is not None:
            scaler.scale(loss).backward()
            scaler.unscale_(optimizer)           # unscale before clipping!
            nn.utils.clip_grad_norm_(model.parameters(), CFG['grad_clip'])
            scaler.step(optimizer)
            scaler.update()
        else:
            loss.backward()
            nn.utils.clip_grad_norm_(model.parameters(), CFG['grad_clip'])
            optimizer.step()

        total_loss += loss.item()
        total_dice += dloss.item()
        total_bce  += bloss.item()

        # Compute per-batch Dice
        with torch.no_grad():
            pred_bin = (torch.sigmoid(preds.detach()) > CFG['threshold']).float()
            all_dice.append(dice_coefficient(pred_bin, masks).item())

        if (step + 1) % 20 == 0:
            print(f'  Ep{epoch} [{step+1}/{n}] loss={loss.item():.4f} dice={all_dice[-1]:.4f}')

    return {
        'loss': total_loss/n, 'dice_loss': total_dice/n, 'bce_loss': total_bce/n,
        'dice': np.mean(all_dice),
    }


@torch.no_grad()
def validate(model, loader):
    """
    Validate on val/test set. @torch.no_grad() disables gradient graph.
    This saves ~50% GPU memory and speeds up inference 2x.
    model.eval() uses running BN statistics (not batch statistics) —
    CRITICAL: forgetting this gives noisy, unreliable validation metrics.
    """
    model.eval()
    total_loss = total_dice_loss = 0.0
    all_dice   = []
    all_iou    = []
    all_sens   = []
    all_spec   = []
    all_prec   = []
    all_probs  = []
    all_labels = []
    n = len(loader)

    for images, masks in loader:
        images = images.to(device, non_blocking=True)
        masks  = masks.to(device,  non_blocking=True)

        preds = model(images)
        loss, dloss, _ = criterion(preds, masks)
        total_loss      += loss.item()
        total_dice_loss += dloss.item()

        probs    = torch.sigmoid(preds)
        pred_bin = (probs > CFG['threshold']).float()

        all_dice.append(dice_coefficient(pred_bin, masks).item())
        all_iou.append(iou_score(pred_bin, masks).item())
        all_sens.append(sensitivity(pred_bin, masks).item())
        all_spec.append(specificity(pred_bin, masks).item())
        all_prec.append(precision_score(pred_bin, masks).item())

        # Collect probabilities for AUC computation
        all_probs.append(probs.cpu().numpy().reshape(-1))
        all_labels.append(masks.cpu().numpy().reshape(-1))

    all_probs_flat  = np.concatenate(all_probs)
    all_labels_flat = np.concatenate(all_labels).astype(int)

    return {
        'loss'       : total_loss / n,
        'dice'       : round(np.mean(all_dice), 4),
        'iou'        : round(np.mean(all_iou), 4),
        'sensitivity': round(np.mean(all_sens), 4),
        'specificity': round(np.mean(all_spec), 4),
        'precision'  : round(np.mean(all_prec), 4),
        'auc_roc'    : round(compute_auc_roc(all_probs_flat, all_labels_flat), 4),
        'auc_pr'     : round(compute_auc_pr(all_probs_flat, all_labels_flat), 4),
        '_probs'     : all_probs_flat,
        '_labels'    : all_labels_flat,
    }


print('Training and validation loops ready')


In [ ]:
# ============================================================
# Cell 13: Checkpointing
# ============================================================

def save_checkpoint(model, optimizer, scheduler, epoch, metrics, filename):
    """
    Save full training state — not just model weights.
    Saving optimizer + scheduler state enables resuming from exact same point
    if the session crashes or runs out of time (common on Colab/Kaggle).
    """
    path = os.path.join(CFG['results_dir'], 'checkpoints', filename)
    torch.save({
        'epoch'                : epoch,
        'model_state_dict'     : model.state_dict(),
        'optimizer_state_dict' : optimizer.state_dict(),
        'scheduler_state_dict' : scheduler.state_dict(),
        'metrics'              : metrics,
        'config'               : CFG,
    }, path)
    print(f'  Checkpoint saved: {filename}  (Dice: {metrics["dice"]:.4f})')


def load_best_model(model, filename='best_model.pt'):
    """Load best checkpoint for evaluation."""
    path = os.path.join(CFG['results_dir'], 'checkpoints', filename)
    ckpt = torch.load(path, map_location=device, weights_only=False)
    model.load_state_dict(ckpt['model_state_dict'])
    print(f'Loaded best model from epoch {ckpt["epoch"]}  (Val Dice: {ckpt["metrics"]["dice"]:.4f})')
    return ckpt['metrics']


print('Checkpointing functions ready')


In [ ]:
# ============================================================
# Cell 14: Main Training — Two-Phase Strategy
# ============================================================

set_seed(CFG['seed'])

scaler    = GradScaler() if CFG['use_amp'] and device.type == 'cuda' else None
history   = []   # stores per-epoch metrics for plotting
best_dice = 0.0
patience_counter = 0

print('=' * 60)
print('PHASE 1: Encoder FROZEN — warming up decoder + attention gates')
print('=' * 60)

# ── PHASE 1: Freeze encoder, train only decoder ──────────────────────────────
freeze_encoder(model)
optimizer = build_optimizer(model, phase=1)
scheduler = build_scheduler(optimizer, CFG['freeze_epochs'], warmup_epochs=0)

for epoch in range(1, CFG['freeze_epochs'] + 1):
    tr = train_one_epoch(model, train_loader, optimizer, criterion, scaler, epoch)
    vl = validate(model, val_loader)
    scheduler.step()
    lr_now = scheduler.get_last_lr()[0]

    row = {'phase': 1, 'epoch': epoch, 'lr': lr_now, **{f'train_{k}': v for k, v in tr.items()}, **{f'val_{k}': v for k, v in vl.items() if not k.startswith('_')}}
    history.append(row)

    print(f'  Ep {epoch:02d}/{CFG["freeze_epochs"]} | '
          f'Train Dice: {tr["dice"]:.4f} | Val Dice: {vl["dice"]:.4f} | '
          f'IoU: {vl["iou"]:.4f} | Sens: {vl["sensitivity"]:.4f} | LR: {lr_now:.2e}')

    if vl['dice'] > best_dice:
        best_dice = vl['dice']
        save_checkpoint(model, optimizer, scheduler, epoch, vl, 'best_model.pt')

print(f'\nPhase 1 complete. Best Val Dice so far: {best_dice:.4f}')

# ── PHASE 2: Unfreeze encoder, full fine-tuning ──────────────────────────────
print('\n' + '='*60)
print('PHASE 2: Encoder UNFROZEN — full fine-tuning with differential LR')
print('='*60)

unfreeze_encoder(model)
optimizer = build_optimizer(model, phase=2)
remaining = CFG['total_epochs'] - CFG['freeze_epochs']
scheduler = build_scheduler(optimizer, remaining, CFG['warmup_epochs'])

patience_counter = 0

for epoch in range(CFG['freeze_epochs'] + 1, CFG['total_epochs'] + 1):
    tr = train_one_epoch(model, train_loader, optimizer, criterion, scaler, epoch)
    vl = validate(model, val_loader)
    scheduler.step()

    # Log both encoder and decoder LR
    lr_enc = optimizer.param_groups[0]['lr']
    lr_dec = optimizer.param_groups[1]['lr']

    row = {'phase': 2, 'epoch': epoch, 'lr_encoder': lr_enc, 'lr_decoder': lr_dec,
           **{f'train_{k}': v for k, v in tr.items()},
           **{f'val_{k}': v for k, v in vl.items() if not k.startswith('_')}}
    history.append(row)

    print(f'  Ep {epoch:02d}/{CFG["total_epochs"]} | '
          f'Train Dice: {tr["dice"]:.4f} | Val Dice: {vl["dice"]:.4f} | '
          f'IoU: {vl["iou"]:.4f} | Sens: {vl["sensitivity"]:.4f} | '
          f'LR enc={lr_enc:.1e} dec={lr_dec:.1e}')

    if vl['dice'] > best_dice:
        best_dice = vl['dice']
        patience_counter = 0
        save_checkpoint(model, optimizer, scheduler, epoch, vl, 'best_model.pt')
    else:
        patience_counter += 1
        if patience_counter >= CFG['patience']:
            print(f'\nEarly stopping at epoch {epoch}. Best Val Dice: {best_dice:.4f}')
            break

    # Save every 10 epochs as backup
    if epoch % 10 == 0:
        save_checkpoint(model, optimizer, scheduler, epoch, vl, f'epoch_{epoch:03d}.pt')

# Save full history
history_path = os.path.join(CFG['results_dir'], 'metrics', 'training_history.json')
with open(history_path, 'w') as f:
    json.dump(history, f, indent=2)

print(f'\nTraining complete. Best Val Dice: {best_dice:.4f}')
print(f'History saved to: {history_path}')


In [ ]:
# ============================================================
# Cell 15: Plot 1 — Training History Curves
# ============================================================

history_path = os.path.join(CFG['results_dir'], 'metrics', 'training_history.json')
with open(history_path) as f:
    hist = json.load(f)

epochs     = [h['epoch']            for h in hist]
train_loss = [h.get('train_loss', 0) for h in hist]
val_loss   = [h.get('val_loss', 0)   for h in hist]
train_dice = [h.get('train_dice', 0) for h in hist]
val_dice   = [h.get('val_dice', 0)   for h in hist]
val_iou    = [h.get('val_iou', 0)    for h in hist]
val_sens   = [h.get('val_sensitivity', 0) for h in hist]
val_spec   = [h.get('val_specificity', 0) for h in hist]
val_auc    = [h.get('val_auc_roc', 0)     for h in hist]

# Find phase boundary
phase_boundary = next((h['epoch'] for h in hist if h.get('phase') == 2), None)

fig, axes = plt.subplots(2, 2, figsize=(16, 12))
fig.suptitle('Attention U-Net — Training History (ResNet34 + scSE)', fontsize=15, fontweight='bold')

# Loss
ax = axes[0, 0]
ax.plot(epochs, train_loss, label='Train loss', color='#2196F3', lw=2)
ax.plot(epochs, val_loss,   label='Val loss',   color='#FF5722', lw=2)
if phase_boundary:
    ax.axvline(x=phase_boundary, color='gray', ls='--', lw=1.5, label='Unfreeze encoder')
ax.set_title('Combined Loss (Dice + BCE)', fontsize=12)
ax.set_xlabel('Epoch'); ax.set_ylabel('Loss')
ax.legend(); ax.grid(True, alpha=0.3)

# Dice
ax = axes[0, 1]
ax.plot(epochs, train_dice, label='Train Dice', color='#2196F3', lw=2)
ax.plot(epochs, val_dice,   label='Val Dice',   color='#FF5722', lw=2)
ax.axhline(y=0.82, color='green', ls='--', lw=1.5, label='Target (0.82)')
if phase_boundary:
    ax.axvline(x=phase_boundary, color='gray', ls='--', lw=1.5, label='Unfreeze')
ax.set_title('Dice Score', fontsize=12)
ax.set_xlabel('Epoch'); ax.set_ylabel('Dice')
ax.legend(); ax.grid(True, alpha=0.3); ax.set_ylim([0, 1])

# All val metrics
ax = axes[1, 0]
ax.plot(epochs, val_dice, label='Dice',        color='#2196F3', lw=2)
ax.plot(epochs, val_iou,  label='IoU',         color='#9C27B0', lw=2)
ax.plot(epochs, val_sens, label='Sensitivity', color='#FF9800', lw=2)
ax.plot(epochs, val_spec, label='Specificity', color='#4CAF50', lw=2)
ax.plot(epochs, val_auc,  label='AUC-ROC',     color='#E91E63', lw=2)
ax.axhline(y=0.82, color='#2196F3', ls=':', alpha=0.5)
ax.axhline(y=0.80, color='#FF9800', ls=':', alpha=0.5)
ax.set_title('All Validation Metrics', fontsize=12)
ax.set_xlabel('Epoch'); ax.set_ylabel('Score')
ax.legend(fontsize=9); ax.grid(True, alpha=0.3); ax.set_ylim([0, 1])

# Loss components
ax = axes[1, 1]
dice_loss_h = [h.get('train_dice_loss', 0) for h in hist]
bce_loss_h  = [h.get('train_bce_loss', 0)  for h in hist]
ax.plot(epochs, dice_loss_h, label='Dice loss component', color='#FF5722', lw=2)
ax.plot(epochs, bce_loss_h,  label='BCE loss component',  color='#9C27B0', lw=2)
ax.set_title('Loss Components (train)', fontsize=12)
ax.set_xlabel('Epoch'); ax.set_ylabel('Loss')
ax.legend(); ax.grid(True, alpha=0.3)

plt.tight_layout()
out = os.path.join(CFG['results_dir'], 'plots', 'training_history.png')
plt.savefig(out, dpi=150, bbox_inches='tight')
plt.show()
print(f'Training curves saved → {out}')


In [ ]:
# ============================================================
# Cell 16: Plot 2 — ROC and Precision-Recall Curves
# ============================================================

# Load best model then collect test set predictions
best_ckpt_metrics = load_best_model(model)

print('Collecting test set predictions for ROC / PR curves...')
test_results = validate(model, test_loader)

probs  = test_results['_probs']
labels = test_results['_labels']

# Sub-sample — pixel-level = millions of values
idx    = np.random.choice(len(probs), min(500_000, len(probs)), replace=False)
probs_s  = probs[idx]
labels_s = labels[idx].astype(int)

# ROC
fpr, tpr, _  = roc_curve(labels_s, probs_s)
roc_auc      = test_results['auc_roc']

# PR
prec, rec, _ = precision_recall_curve(labels_s, probs_s)
auc_pr       = test_results['auc_pr']
vessel_ratio = labels_s.mean()

fig, axes = plt.subplots(1, 2, figsize=(14, 6))
fig.suptitle('Attention U-Net — ROC and PR Curves (Test Set)', fontsize=14, fontweight='bold')

# ROC curve
ax = axes[0]
ax.plot(fpr, tpr, color='#2196F3', lw=2.5, label=f'AUC-ROC = {roc_auc:.4f}')
ax.plot([0, 1], [0, 1], color='gray', ls='--', lw=1)
ax.fill_between(fpr, tpr, alpha=0.08, color='#2196F3')
ax.set_xlim([0, 1]); ax.set_ylim([0, 1.02])
ax.set_xlabel('False Positive Rate  (1 - Specificity)', fontsize=11)
ax.set_ylabel('True Positive Rate  (Sensitivity)', fontsize=11)
ax.set_title('ROC Curve', fontsize=13)
ax.legend(loc='lower right', fontsize=11)
ax.grid(True, alpha=0.3)

# PR curve
ax = axes[1]
ax.plot(rec, prec, color='#FF5722', lw=2.5, label=f'AUC-PR = {auc_pr:.4f}')
ax.axhline(y=vessel_ratio, color='gray', ls='--', lw=1,
           label=f'Random baseline ({vessel_ratio:.3f})')
ax.fill_between(rec, prec, alpha=0.08, color='#FF5722')
ax.set_xlim([0, 1]); ax.set_ylim([0, 1.02])
ax.set_xlabel('Recall  (Sensitivity)', fontsize=11)
ax.set_ylabel('Precision', fontsize=11)
ax.set_title('Precision-Recall Curve', fontsize=13)
ax.legend(loc='upper right', fontsize=11)
ax.grid(True, alpha=0.3)

plt.tight_layout()
out = os.path.join(CFG['results_dir'], 'plots', 'roc_pr_curves.png')
plt.savefig(out, dpi=150, bbox_inches='tight')
plt.show()
print(f'ROC/PR curves saved → {out}')
print(f'AUC-ROC: {roc_auc:.4f}  |  AUC-PR: {auc_pr:.4f}')


In [ ]:
@torch.no_grad()
def visualize_predictions(model, loader, n_samples=8, save_dir=None):
    model.eval()
    save_dir = save_dir or os.path.join(CFG['results_dir'], 'predictions')
    os.makedirs(save_dir, exist_ok=True)
    MEAN = np.array([0.485, 0.456, 0.406])
    STD  = np.array([0.229, 0.224, 0.225])
    count = 0

    for images, masks in loader:

        if count >= n_samples:
            break

        images_gpu = images.to(device, non_blocking=True)
        probs_gpu  = torch.sigmoid(model(images_gpu))

        probs_np   = probs_gpu.cpu().numpy()
        images_np  = images.numpy()
        masks_np   = masks.numpy()

        for i in range(min(images.size(0), n_samples - count)):

            # Undo ImageNet normalization for display
            img = images_np[i].transpose(1, 2, 0)   # (C,H,W) -> (H,W,C)
            img = np.clip(img * STD + MEAN, 0, 1)

            gt   = masks_np[i, 0]
            prob = probs_np[i, 0]
            pred = (prob > CFG['threshold']).astype(float)

            # Overlay: red vessels on original image
            overlay = img.copy()
            overlay[pred > 0.5] = [1.0, 0.15, 0.15]

            # Error map: green=TP, red=FP, blue=FN
            err = np.zeros((*gt.shape, 3))
            err[..., 1] = pred * gt
            err[..., 0] = pred * (1 - gt)
            err[..., 2] = (1 - pred) * gt

            # Metrics
            inter      = (pred * gt).sum()
            img_dice   = (2 * inter + 1) / (pred.sum() + gt.sum() + 1)
            img_sens   = inter / (gt.sum() + 1e-6)
            vessel_pct = gt.mean() * 100

            fig, axes = plt.subplots(1, 5, figsize=(25, 5))
            fig.suptitle(
                f'Sample {count+1} | Dice: {img_dice:.4f} | '
                f'Sensitivity: {img_sens:.4f} | Vessel pixels: {vessel_pct:.2f}%',
                fontsize=12, fontweight='bold'
            )

            axes[0].imshow(img)
            axes[0].set_title('Original fundus image')
            axes[0].axis('off')

            axes[1].imshow(gt, cmap='gray')
            axes[1].set_title('Ground truth mask')
            axes[1].axis('off')

            im = axes[2].imshow(prob, cmap='hot', vmin=0, vmax=1)
            axes[2].set_title('Prediction probability')
            axes[2].axis('off')
            plt.colorbar(im, ax=axes[2], fraction=0.046, pad=0.04)

            axes[3].imshow(overlay)
            axes[3].set_title('Prediction overlay (vessels=red)')
            axes[3].axis('off')

            axes[4].imshow(err)
            tp_p = mpatches.Patch(color='green', label='TP')
            fp_p = mpatches.Patch(color='red',   label='FP')
            fn_p = mpatches.Patch(color='blue',  label='FN')
            axes[4].legend(handles=[tp_p, fp_p, fn_p], loc='lower right', fontsize=8)
            axes[4].set_title('Error map')
            axes[4].axis('off')

            plt.tight_layout()
            out_path = os.path.join(save_dir, f'prediction_{count+1:02d}.png')
            plt.savefig(out_path, dpi=120, bbox_inches='tight')
            plt.show()
            plt.close()

            count += 1

    print(f'{count} prediction visualizations saved → {save_dir}')


visualize_predictions(model, test_loader, n_samples=8)


In [ ]:
import os
import json
import numpy as np
import torch
from PIL import Image

# ------------------------------------------------------------
# Helper: convert NumPy / Torch values to JSON-safe Python types
# ------------------------------------------------------------
def to_serializable(obj):
    if isinstance(obj, dict):
        return {k: to_serializable(v) for k, v in obj.items()}
    elif isinstance(obj, list):
        return [to_serializable(v) for v in obj]
    elif isinstance(obj, tuple):
        return tuple(to_serializable(v) for v in obj)
    elif isinstance(obj, np.ndarray):
        return obj.tolist()
    elif isinstance(obj, (np.float32, np.float64, np.float16)):
        return float(obj)
    elif isinstance(obj, (np.int32, np.int64, np.int16, np.int8)):
        return int(obj)
    elif isinstance(obj, (np.bool_,)):
        return bool(obj)
    elif isinstance(obj, torch.Tensor):
        if obj.numel() == 1:
            return obj.item()
        return obj.detach().cpu().tolist()
    return obj


# ------------------------------------------------------------
# Final results dictionary
# ------------------------------------------------------------
final = {
    'model_info': {
        'architecture'   : 'Attention U-Net',
        'encoder'        : CFG['encoder_name'],
        'attention_type' : CFG['attention_type'],
        'pretrained_on'  : CFG['encoder_weights'],
        'img_size'       : CFG['img_size'],
        'n_train'        : len(train_dataset),
        'n_val'          : len(val_dataset),
        'n_test'         : len(test_dataset),
    },
    'hyperparameters': {
        'lr_encoder'     : CFG['lr_encoder'],
        'lr_decoder'     : CFG['lr_decoder'],
        'batch_size'     : CFG['batch_size'],
        'freeze_epochs'  : CFG['freeze_epochs'],
        'total_epochs'   : CFG['total_epochs'],
        'dice_weight'    : CFG['dice_weight'],
        'bce_weight'     : CFG['bce_weight'],
        'label_smoothing': CFG['label_smoothing'],
    },
    'test_metrics': {
        'dice'       : test_results['dice'],
        'iou'        : test_results['iou'],
        'sensitivity': test_results['sensitivity'],
        'specificity': test_results['specificity'],
        'precision'  : test_results['precision'],
        'auc_roc'    : test_results['auc_roc'],
        'auc_pr'     : test_results['auc_pr'],
    },
    'targets': {
        'dice_target'       : 0.82,
        'dice_met'          : test_results['dice'] >= 0.82,
        'iou_target'        : 0.75,
        'iou_met'           : test_results['iou'] >= 0.75,
        'sensitivity_target': 0.80,
        'sensitivity_met'   : test_results['sensitivity'] >= 0.80,
        'specificity_target': 0.97,
        'specificity_met'   : test_results['specificity'] >= 0.97,
    },
}

# Convert to JSON-safe object
final_safe = to_serializable(final)

# Make output folders
metrics_dir = os.path.join(CFG['results_dir'], 'metrics')
os.makedirs(metrics_dir, exist_ok=True)

out_path = os.path.join(metrics_dir, 'test_results.json')
with open(out_path, 'w') as f:
    json.dump(final_safe, f, indent=2)

# Pretty print
print('=' * 58)
print('  ATTENTION U-NET  —  FINAL TEST SET RESULTS')
print('=' * 58)
print(f'  Dice Score   : {float(final_safe["test_metrics"]["dice"]):.4f}   (target > 0.82)  {"✅" if final_safe["targets"]["dice_met"] else "❌"}')
print(f'  IoU          : {float(final_safe["test_metrics"]["iou"]):.4f}   (target > 0.75)  {"✅" if final_safe["targets"]["iou_met"] else "❌"}')
print(f'  Sensitivity  : {float(final_safe["test_metrics"]["sensitivity"]):.4f}   (target > 0.80)  {"✅" if final_safe["targets"]["sensitivity_met"] else "❌"}')
print(f'  Specificity  : {float(final_safe["test_metrics"]["specificity"]):.4f}   (target > 0.97)  {"✅" if final_safe["targets"]["specificity_met"] else "❌"}')
print(f'  Precision    : {float(final_safe["test_metrics"]["precision"]):.4f}')
print(f'  AUC-ROC      : {float(final_safe["test_metrics"]["auc_roc"]):.4f}')
print(f'  AUC-PR       : {float(final_safe["test_metrics"]["auc_pr"]):.4f}')
print('=' * 58)
print(f'\n  Results saved → {out_path}')

# List all saved files
print('\n  attention_UNet_results/ — all saved files:')
for root, dirs, files in os.walk(CFG['results_dir']):
    for file in sorted(files):
        fpath = os.path.join(root, file)
        size  = os.path.getsize(fpath)
        rel   = os.path.relpath(fpath, CFG['results_dir'])
        size_str = f'{size/1024:.0f} KB' if size < 1e6 else f'{size/1e6:.1f} MB'
        print(f'    {rel:<45} {size_str}')

In [ ]:
import shutil

folder_path = "attention_UNet_results"   # <-- your folder name
output_zip = "attention_UNet_results.zip"

shutil.make_archive("attention_UNet_results", 'zip', folder_path)

print("ZIP created:", output_zip)